# <span style="color:#F57C00">Exploratory Data Analysis: What the District Tables Tell Us Before We Build the Model</span>

We now have six tables, all joined through the **district code** (`district_code`; district names repeat, e.g. there are 100 districts called like another one, so we never join on names):

| Table | One row per | Years |
| :--- | :--- | :--- |
| greenness (Landsat) | district and year | 2013-2025, July + August |
| climate (ERA5) | district and month | 2000-2025 |
| population (TÜİK) | district and year | 2013-2025 |
| building permits (TÜİK) | district and month | 2013-2025 |
| static table (GHSL, SRTM) | district | - |
| SEGE-2022 | district | 2022 only |

The model will predict the **summer surface temperature** of the built-up area of a district (Landsat). This notebook answers one question that decides how the model must be trained and tested. Every part can be run on its own after the setup, and ends with a short **What we learned**.

| Part | Question |
| :--- | :--- |
| **A2** | Which districts are left out because their satellite value is not reliable, and does this bias the picture? |

**Two words used a lot.** *Reliable* row = a district-year with a built-up area of at least 0.5 km² and with a clear view in at least half of its built-up pixels. *Within city* = a value minus the mean of the same city in the same year: what is left is the difference between the districts of one city, so weather and regional climate drop out. This is the part that parks and buildings can change.

# <span style="color:#F57C00">Setup: Load the Tables and Build One Analysis Table</span>

We read the processed tables and build **one table with one row per district and summer** (973 districts x 13 years). The rule: each source table is aggregated to this level first and then joined on `district_code` (and `year`). `validate=` makes pandas stop with an error if a join would repeat rows.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import sparse
from scipy.stats import ks_2samp
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, roc_auc_score
from sklearn.model_selection import GroupKFold, KFold, LeaveOneGroupOut, cross_val_predict

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100

PROCESSED = Path("../data/processed")

green = pd.read_csv(PROCESSED / "greenness_districts_summer_2013_2025.csv")
climate = pd.read_csv(PROCESSED / "climate_districts_monthly_2000_2025.csv")
population = pd.read_csv(PROCESSED / "tuik_districts_population_clean.csv")
permits = pd.read_csv(PROCESSED / "building_permits_districts_monthly_2013_2025.csv")
static = pd.read_csv(PROCESSED / "districts_urban_static.csv")
sege = pd.read_csv(PROCESSED / "sege_districts_2022.csv")
districts = gpd.read_file(PROCESSED / "turkey_districts_master.geojson")
cities = gpd.read_file(PROCESSED / "turkey_cities_master.geojson")

for name, table in [("greenness", green), ("climate", climate), ("population", population), ("permits", permits), ("static", static), ("sege", sege)]:
    print(f"{name:11s} {table.shape[0]:>7} rows x {table.shape[1]:>2} columns")


## Step 1: Two groupings: metropolitan cities and regions
Two simple groupings help us to read the results: the **30 metropolitan municipalities** (`city_code` = the plate number of the province; in them the TÜİK urban population is the whole population, outside them it is only the district center) and the **7 geographic regions**.

In [ ]:
METRO = {1, 6, 7, 9, 10, 16, 20, 21, 25, 26, 27, 31, 33, 34, 35, 38, 41, 42, 44, 45, 46, 47, 48, 52, 54, 55, 59, 61, 63, 65}

REGIONS = {
    "Marmara": [10, 11, 16, 17, 22, 34, 39, 41, 54, 59, 77],
    "Aegean": [3, 9, 20, 35, 43, 45, 48, 64],
    "Mediterranean": [1, 7, 15, 31, 32, 33, 46, 80],
    "Central Anatolia": [6, 18, 26, 38, 40, 42, 50, 51, 58, 66, 68, 70, 71],
    "Black Sea": [5, 8, 14, 19, 28, 29, 37, 52, 53, 55, 57, 60, 61, 67, 69, 74, 78, 81],
    "Eastern Anatolia": [4, 12, 13, 23, 24, 25, 30, 36, 44, 49, 62, 65, 75, 76],
    "Southeastern Anatolia": [2, 21, 27, 47, 56, 63, 72, 73, 79],
}
REGION_OF = {city: region for region, city_list in REGIONS.items() for city in city_list}

assert len(METRO) == 30 and len(REGION_OF) == 81, "30 metropolitan cities and 81 provinces expected"


## Step 2: Summer weather, spring rain, permits and population per district and year
- **Summer weather:** mean temperature of July + August (`temp_mean_ja`) and the number of hot days (≥ 30 °C) in July + August (`hot_days_ja`). This is the *weather of the same summer* as the satellite picture.
- **Spring rain** (March-June, `precip_spring`): dry springs may mean less green in summer.
- **Permits:** the sum of the 12 months before the end of June, i.e. July of the year before until June (`permit_12m`). One month alone is too spiky. The first summer (2013) has no full 12 months, so it stays empty.
- **Population:** share of people aged 65+ (`share65`), the logarithm of the population (`logpop`: the size of the district, on a scale where a district twice as big is one step more) and the yearly growth (`pop_growth`).

In [ ]:
summer_weather = (climate[climate["month"].isin([7, 8])].groupby(["district_code", "year"])
                  .agg(temp_mean_ja=("temp_mean", "mean"), hot_days_ja=("hot_days", "sum")).reset_index())
spring_rain = (climate[climate["month"].between(3, 6)].groupby(["district_code", "year"])
               .agg(precip_spring=("precip_sum", "sum")).reset_index())

permits = permits.sort_values(["district_code", "year", "month"])
permits["permit_12m"] = permits.groupby("district_code")["permit_area_total_m2"].transform(lambda s: s.rolling(12).sum())
permit_june = permits.loc[permits["month"] == 6, ["district_code", "year", "permit_12m"]]  # the 12 months up to June

# The age groups from 65 on: total_65_69, total_70_74, ..., total_90plus
old_columns = [c for c in population.columns if c.startswith("total_") and c.split("_")[1][:2] in ("65", "70", "75", "80", "85", "90")]
population = population.sort_values(["district_code", "year"])
population["share65"] = population[old_columns].sum(axis=1) / population["total_pop"]
population["logpop"] = np.log(population["total_pop"])
population["pop_growth"] = population.groupby("district_code")["total_pop"].pct_change()

print("age groups counted as 65+:", [c.replace("total_", "") for c in old_columns])


## Step 3: Join everything on the district code
The greenness table is the base (one row per district and year). The static table and SEGE have one row per district, so they are joined with `many_to_one`.
The last lines add the *reliable* flag and the groupings of Step 1.

In [ ]:
df = (green
      .merge(summer_weather, on=["district_code", "year"], how="left", validate="one_to_one")
      .merge(spring_rain, on=["district_code", "year"], how="left", validate="one_to_one")
      .merge(permit_june, on=["district_code", "year"], how="left", validate="one_to_one")
      .merge(population[["district_code", "year", "total_pop", "share65", "logpop", "pop_growth"]], on=["district_code", "year"], how="left", validate="one_to_one")
      .merge(static.drop(columns=["city_code", "district_name", "city_name"]), on="district_code", how="left", validate="many_to_one")
      .merge(sege[["district_code", "sege_score", "sege_tier"]], on="district_code", how="left", validate="many_to_one")
      .merge(districts[["district_code", "center_lat", "center_lon"]], on="district_code", how="left", validate="many_to_one"))

df["permit_12m_per_built"] = df["permit_12m"] / df["built_m2_2020"]  # new floor area per m² of built-up ground (flow relative to stock)
df["metro"] = df["city_code"].isin(METRO)
df["region"] = df["city_code"].map(REGION_OF)
df["reliable"] = (df["urban_km2"] >= 0.5) & (df["landsat_cover"] >= 0.5) & df["landsat_lst_c"].notna()

assert len(df) == len(green) == 973 * 13, "the joins must not change the number of rows"
print(f"{len(df)} district-years | reliable: {df['reliable'].sum()} ({df['reliable'].mean():.0%})")
print("columns that are empty somewhere:", df.isna().sum()[lambda s: s > 0].to_dict())


## Step 4: Helpers for "within city" and the comparable rows
`minus_city_year` subtracts, for every column, the mean of the same city in the same year.
Comparing districts inside a city only makes sense if the city has at least 3 reliable districts in that year: `within` keeps only those rows (about 10,600 of the 10,657 reliable rows).

In [ ]:
def minus_city_year(table, columns):
    # value minus the mean of the same city and year: weather and regional climate drop out
    return table[columns] - table.groupby(["city_code", "year"])[columns].transform("mean")


reliable = df[df["reliable"]].copy()
cities_per_year = reliable.groupby(["city_code", "year"])["district_code"].transform("size")
within = reliable[cities_per_year >= 3].copy()
print(f"reliable rows: {len(reliable)} | rows that can be compared inside a city: {len(within)}")


# <span style="color:#F57C00">A2: Which Districts Are Left Out, and Does It Matter?</span>

The satellite value of a district is only used if its built-up area is at least 0.5 km² and at least half of its built-up pixels had a clear view. The other rows are left out of the model. The risk: if the left-out districts are special (poor, rural, in one region), the model and the planning results tell nothing about them.

## Step 1: Why are rows left out?

In [ ]:
df["status"] = np.select(
    [df["landsat_lst_c"].isna(), df["urban_km2"] < 0.5, df["landsat_cover"] < 0.5],
    ["no Landsat value", "built-up area under 0.5 km²", "under half of the pixels clear"],
    default="reliable")

counts = df["status"].value_counts()
print(counts.to_string())
counts.drop("reliable").plot.barh(figsize=(7, 2.2), color="#F57C00", title="Why district-years are left out")
plt.gca().invert_yaxis()
plt.show()


## Step 2: Who is left out? Metropolitan or not, region, development level
For every group we show the share of district-years that are left out. The SEGE tier is 1 (most developed) to 6 (least developed).

In [ ]:
df["left_out"] = ~df["reliable"]
df["metro_label"] = np.where(df["metro"], "Metropolitan", "Other provinces")

fig, axes = plt.subplots(1, 3, figsize=(16, 4), gridspec_kw={"width_ratios": [1, 2.2, 1.6]})
for ax, group, title in zip(axes, ["metro_label", "region", "sege_tier"], ["Metropolitan or not", "Region", "SEGE tier (1 = most developed)"]):
    share = df.groupby(group)["left_out"].mean().sort_values(ascending=False) if group == "region" else df.groupby(group)["left_out"].mean()
    ax.bar(share.index.astype(str), share.values, color="#F57C00")
    ax.set_title(title); ax.set_ylabel("share left out"); ax.set_ylim(0, 0.4)
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


## Step 3: How big are the left-out districts, and how many people live in them?
Population is the urban TÜİK population of 2025. A box plot with a logarithmic axis shows both small and big districts.

In [ ]:
latest = df[df["year"] == 2025].copy()
latest["group"] = np.where(latest["reliable"], "reliable in 2025", "left out in 2025")

fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=latest, x="group", y="total_pop", hue="group", palette=["#F57C00", "#7aa6c2"], ax=ax, legend=False)
ax.set_yscale("log"); ax.set_ylabel("urban population in 2025 (log scale)"); ax.set_xlabel("")
plt.show()

print("median urban population:", latest.groupby("group")["total_pop"].median().round(0).to_dict())
share_people = latest.loc[~latest["reliable"], "total_pop"].sum() / latest["total_pop"].sum()
print(f"share of the urban population that lives in left-out districts: {share_people:.1%}")

district_status = df.groupby("district_code")["reliable"].mean()
print(f"districts: never reliable {int((district_status == 0).sum())} | always reliable {int((district_status == 1).sum())} | in between {int(((district_status > 0) & (district_status < 1)).sum())}")


**What we learned (A2)**
- Most rows are left out because the **built-up area is under 0.5 km²** (1,841 of the 1,992 left-out district-years). Bad weather (clouds) hardly matters (144 rows).
- The left-out districts are **small and in the east and north**: about a third of the district-years of the Black Sea (33%) and of Eastern Anatolia (29%) are left out, against 5-8% in Marmara, the Aegean and Central Anatolia. In metropolitan provinces 12% are left out, elsewhere 20%.
- They are also **the least developed ones**: 31% (tier 5) and 30% (tier 6) of their district-years are left out, against 1-8% in tiers 1-3.
- But they hold **only 1.3% of the urban population** (median 4,851 people against 30,266). For the question "where to plant green to cool the most people" this loses little. For a statement about equity ("poor districts are hotter") it matters: a third of the poorest tiers is invisible, and the report must say so.
- 142 districts are never reliable, 742 always, 89 sometimes. The share left out is almost the same in every year (15-19%), so the rule does not bias the time series.